In [133]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
import math
from collections import Counter

from ultralytics.utils.checks import check_imshow
from ultralytics.utils.plotting import Annotator, colors

from collections import defaultdict

In [134]:
# COLORS = sv.ColorPalette.from_hex(["#E6194B", "#3CB44B", "#FFE119", "#3C76D1"])
COLORS = sv.ColorPalette.from_hex(["#00FF00", "#FF0000"])

ZONE_IN_POLYGONS = [
    np.array([[1021, 969], [1154, 982], [1218, 825], [1088, 808]]),
    np.array([[1574, 514], [1583, 459], [1378, 402], [1359, 453]]),
    np.array([[995, 227], [923, 211], [834, 268], [921, 288]]),
    np.array([[393, 540], [449, 467], [549, 507], [480, 593]]),
]

ZONE_OUT_POLYGONS = [
    np.array([[759, 962], [921, 951], [863, 806], [714, 823]]),
    np.array([[1419, 562], [1433, 654], [1586, 619], [1547, 536]]),
    np.array([[1063, 290], [1156, 271], [1138, 197], [1029, 207]]),
    np.array([[462, 452], [456, 366], [624, 347], [607, 434]]),
]

    # Entradas vuelo01
    # np.array([[783, 1041], [575, 1017], [831, 836], [1002, 925]]),
    # np.array([[1510, 579], [1509, 509], [1902, 646], [1886, 765]]),
    # np.array([[1169, 356], [1068, 310], [1225, 204], [1325, 242]]),
    # np.array([[582, 451], [492, 484], [353, 388], [481, 361]]),

    # Salidas vuelo01
    # np.array([[339, 940], [472, 993], [604, 763], [433, 730]]),
    # np.array([[1491, 830], [1500, 732], [1869, 772], [1844, 885]]),
    # np.array([[1263, 355], [1350, 375], [1408, 251], [1342, 230]]),
    # np.array([[673, 388], [705, 335], [459, 270], [419, 342]]),
    
    # Entradas vuelo03
    # np.array([[1021, 969], [1154, 982], [1218, 825], [1088, 808]]),
    # np.array([[1574, 514], [1583, 459], [1378, 402], [1359, 453]]),
    # np.array([[995, 227], [923, 211], [834, 268], [921, 288]]),
    # np.array([[393, 540], [449, 467], [549, 507], [480, 593]]),

    # Salidas vuelo01
    # np.array([[759, 962], [921, 951], [863, 806], [714, 823]]),
    # np.array([[1419, 562], [1433, 654], [1586, 619], [1547, 536]]),
    # np.array([[1063, 290], [1156, 271], [1138, 197], [1029, 207]]),
    # np.array([[462, 452], [456, 366], [624, 347], [607, 434]]),

model = YOLO("best.pt")
class_names = model.model.names

names = list(class_names.values())
for i in range(len(names)):
    names.append("indeterminado")

# inicializo en cero los dos arrays (de entrada y salida) que tendrán la cantidad de objetos finales por zona y por clase
"""
Example:
    {
        0: {
            "bicycle": 0,
            "bus": 0,
            "car": 0,
            "motorbike": 0,
            "truck": 0,
            "van": 0
        },
        .
        .
        .
        3: {
            "bicycle": 0,
            "bus": 0,
            "car": 0,
            "motorbike": 0,
            "truck": 0,
            "van": 0
        }
    }
"""
total_obj_zone_in = { i: {key: 0 for key in names} for i in range(len(ZONE_IN_POLYGONS)) }
total_obj_zone_out = { i: {key: 0 for key in names} for i in range(len(ZONE_OUT_POLYGONS)) }

"""
classes = {
    "car": "Auto",
    "bus": "Colectivo",
    "light_truck": "Camión liviano",
    "heavy_truck": "Camión pesado",
    "motorbike": "Moto",
    "bicycle": "Bicicleta",
}
"""

'\nclasses = {\n    "car": "Auto",\n    "bus": "Colectivo",\n    "light_truck": "Camión liviano",\n    "heavy_truck": "Camión pesado",\n    "motorbike": "Moto",\n    "bicycle": "Bicicleta",\n}\n'

In [135]:
obj_zones_in = []
obj_zones_out = []
obj_in_for_zones = defaultdict(lambda: [])
obj_out_for_zones = defaultdict(lambda: [])
obj_in_out_zones = defaultdict(lambda: [])
track_results = defaultdict(lambda: [])

track_history = defaultdict(lambda: [])
data_obj_history = defaultdict(lambda: [])

def get_center_bb(box):
    x_center = int((box[0] + box[2]) / 2)
    y_center = int((box[1] + box[3]) / 2)
    return (x_center, y_center)

def draw_polygons(annotated_frame, polygon, number_polygon, zone_type, thickness):
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    cv2.polylines(
        annotated_frame, [polygon], isClosed=True, color=COLORS.colors[zone_type].as_bgr(), thickness=thickness
    )
    zone_center = sv.get_polygon_center(polygon=polygon)
    cv2.putText(annotated_frame, str(number_polygon), (int(zone_center.x), int(zone_center.y)), font, font_scale, COLORS.colors[zone_type].as_bgr(), thickness=thickness)
    
    return annotated_frame

def draw_zones_in_out(annotated_frame, thickness):
    for i, (zone_in, zone_out) in enumerate(zip(ZONE_IN_POLYGONS, ZONE_OUT_POLYGONS)):
        draw_polygons(annotated_frame, zone_in, i, 0, thickness)
        draw_polygons(annotated_frame, zone_out, i, 1, thickness)
    return annotated_frame

def detect_zone_in(box):
    for i, (polygon) in enumerate(ZONE_IN_POLYGONS):
        if (cv2.pointPolygonTest(polygon, get_center_bb(box), False) > 0):  # > 0 dentro del polígono
            return i

    return -1

def detect_zone_out(box):
    for i, (polygon) in enumerate(ZONE_OUT_POLYGONS):
        if (cv2.pointPolygonTest(polygon, get_center_bb(box), False) > 0):  # > 0 dentro del polígono
            return i

    return -1

def save_zone_in(box, track_id):
    if track_id not in obj_zones_in:
        zone_in = detect_zone_in(box)
        if zone_in >= 0:
            obj_zones_in.append(track_id)
            obj_in_for_zones[zone_in].append(track_id)

def save_zone_out(box, track_id):
    if track_id not in obj_zones_out:
        zone_out = detect_zone_out(box)
        if zone_out >= 0:
            obj_zones_out.append(track_id)
            obj_out_for_zones[zone_out].append(track_id)

                
def draw_bb_and_save_track(frame, annotator, box, cls, track_id, act_frame, confidence):
    annotator.box_label(box, color=colors(int(cls), True), label=f"{track_id} - {class_names[int(cls)]}")

    # Store tracking and data object history
    data_obj_history[track_id].append(
        {
            "act_frame": act_frame,
            "class_id": int(cls),
            "confidence": confidence
        })
    track = track_history[track_id]
    track.append((int((box[0] + box[2]) / 2), int((box[1] + box[3]) / 2)))
    if len(track) > 30:
        track.pop(0)

    # Plot tracks
    points = np.array(track, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(frame, [points], isClosed=False, color=colors(int(cls), True), thickness=2)


def total_class_counts(track_data):
    class_counts = Counter()
    for item in track_data:
        class_counts[item['class_id']] += 1
    
    return class_counts

def classify_track(class_counts):
    assigned_class = max(class_counts, key=class_counts.get)
    # Retorna la clase con mayor confianza promedio
    return class_names[assigned_class]

def get_final_results():
    for track_id, data in data_obj_history.items():
        class_counts = total_class_counts(data)
        classification = classify_track(class_counts)
        track_results[track_id] = {
            "classification": classification
        }


In [136]:
video_path = "../vuelo03_1920.mp4"
cap = cv2.VideoCapture(video_path)

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

act_frame = 0

while cap.isOpened() and act_frame < 9000:
    success, frame = cap.read()

    act_frame += 1
    if act_frame % 2 == 0:
        if success:
            results = model.track(
                frame, 
                persist=True, 
                verbose=False,
                conf=0.3,
                iou=0.6,
                agnostic_nms=True, 
                augment=True, 
                device="0"
            )
            boxes = results[0].boxes.xyxy.cpu()

            draw_zones_in_out(frame, 2)
            
            if results[0].boxes.id is not None:
                clss = results[0].boxes.cls.cpu().tolist()
                track_ids = results[0].boxes.id.int().cpu().tolist()
                confs = results[0].boxes.conf.float().cpu().tolist()
                # Annotator Init
                annotator = Annotator(frame, line_width=1)
                for box, cls, track_id, confidence in zip(boxes, clss, track_ids, confs):
                    save_zone_in(box, track_id)
                    save_zone_out(box, track_id)
                    draw_bb_and_save_track(frame, annotator, box, cls, track_id, act_frame, confidence)
            cv2.putText(frame, f"Frame: {act_frame}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.imshow("Video", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        else:
            break

cap.release()
cv2.destroyAllWindows()

get_final_results()

In [137]:
print(obj_in_for_zones.items())
for zone, tracks in obj_in_for_zones.items():
    for track in tracks:
        print(track)
        total_obj_zone_in[zone][track_results[track]["classification"]] += 1
    # print(f"    {zone}: {tracks}")
print(total_obj_zone_in)

dict_items([(3, [11, 18, 33, 17, 50, 206, 305, 343, 348, 371, 396, 410, 401, 419, 437, 463, 510, 599, 515, 411, 542, 491, 653, 519, 541, 691, 893, 935, 1037, 929, 971, 1081, 1136, 1152, 1175, 1301, 1334, 1359, 1379, 1053, 1462, 1586, 1614, 1620, 1632, 1681, 1747, 1818, 1876, 1885, 1887, 1910, 1935, 1954, 1998, 2027, 2159, 2201, 2233, 2286, 2343]), (0, [13, 145, 171, 186, 205, 260, 312, 326, 331, 360, 394, 402, 409, 415, 438, 456, 472, 500, 533, 594, 604, 617, 652, 701, 798, 807, 890, 1071, 1069, 1082, 1167, 1179, 1182, 1186, 1198, 1200, 1240, 1274, 1325, 1350, 1372, 1376, 1418, 1446, 1467, 1486, 1568, 1618, 1706, 1827, 1849, 1884, 1963, 2000, 2061, 2064, 2091, 2148, 2163, 2165, 2193, 2210, 2220, 2232, 2277, 2303, 2374, 2432]), (2, [25, 29, 19, 30, 46, 98, 66, 111, 89, 203, 207, 162, 137, 191, 217, 232, 249, 239, 246, 332, 292, 302, 323, 335, 373, 380, 426, 444, 439, 524, 450, 531, 464, 543, 477, 564, 496, 498, 527, 516, 548, 566, 600, 603, 612, 630, 635, 668, 809, 708, 760, 678, 786, 7

In [ ]:
# Crear un diccionario para contar vehículos por zona de entrada y salida
conteo_por_ruta = {}

print(track_results)
print("obj_zones_in:", obj_zones_in)
print("obj_zones_out:", obj_zones_out) 
# print("obj_in_for_zones:", dict(obj_in_for_zones))
# print("obj_out_for_zones:", dict(obj_out_for_zones))
# print("obj_in_out_zones:", dict(obj_in_out_zones))
# print("track_results:", dict(track_results))

# Recorrer los vehículos detectados
for track_id, info in track_results.items():
    # Obtener zonas de entrada y salida del vehículo
    zone_in = 4
    zone_out = 4
    # Verificar si el vehículo está en las zonas de entrada
    if track_id in obj_zones_in:
        # Si está en las zonas de entrada, obtener la zona de entrada
        for zone, track_ids in obj_in_for_zones.items():
            if track_id in track_ids:
                zone_in = zone
                break
    # Verificar si el vehículo está en las zonas de salida
    if track_id in obj_zones_out:
        # Si está en las zonas de salida, obtener la zona de salida
        for zone, track_ids in obj_out_for_zones.items():
            if track_id in track_ids:
                zone_out = zone
                break
    print(track_id, zone_in, "zone_out:", zone_out)
    if zone_in is not None and zone_out is not None:
        clase = info["classification"]
        
        # Inicializar el diccionario si la ruta no existe
        conteo_por_ruta[zone_in][zone_out] = {
            "car": 0,
            "motorbike": 0,
            "light_truck": 0,
            "heavy_truck": 0,
            "bus": 0,
            "bicycle": 0
        }
        
        # Incrementar el contador para esa clase en esa ruta
        conteo_por_ruta[zone_in][zone_out][clase] += 1

# Mostrar resultados
print("\nConteo de vehículos por ruta (entrada -> salida):")
print("-" * 50)
for (entrada, salida), conteo in conteo_por_ruta.items():
    print(f"\nDe Zona {entrada} a Zona {salida}:")
    for clase, cantidad in conteo.items():
        print(f"    {clase}: {cantidad}")   


defaultdict(<function <lambda> at 0x000002C0742A4E00>, {1: {'classification': 'light_truck'}, 2: {'classification': 'car'}, 3: {'classification': 'car'}, 4: {'classification': 'car'}, 5: {'classification': 'car'}, 6: {'classification': 'car'}, 7: {'classification': 'car'}, 8: {'classification': 'light_truck'}, 9: {'classification': 'car'}, 10: {'classification': 'car'}, 11: {'classification': 'car'}, 12: {'classification': 'car'}, 13: {'classification': 'car'}, 14: {'classification': 'car'}, 15: {'classification': 'car'}, 16: {'classification': 'car'}, 17: {'classification': 'car'}, 18: {'classification': 'car'}, 19: {'classification': 'bus'}, 20: {'classification': 'car'}, 21: {'classification': 'car'}, 22: {'classification': 'car'}, 23: {'classification': 'car'}, 24: {'classification': 'car'}, 25: {'classification': 'car'}, 26: {'classification': 'car'}, 27: {'classification': 'car'}, 28: {'classification': 'car'}, 29: {'classification': 'car'}, 30: {'classification': 'car'}, 31: {'c

In [141]:
# Vehículos	Entrada A				Entrada B				Entrada C				Entrada D			
# 	Salida A	Salida B	Salida C	Salida D	Salida A	Salida B	Salida C	Salida D	Salida A	Salida B	Salida C	Salida D	Salida A	Salida B	Salida C	Salida D
# Auto	0	5	35	11	2	1	19	25	78	13	4	19	8	24	16	0
# Moto	0	0	3	2	1	1	1	1	2	1		3	1	2	2	0
# Camión liviano	0	0	2	2	0	0	0	1	3	0	0	0	0	0	0	0
# Camión pesado	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0
# Colectivo	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0
# Bicicleta	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0
# Total	0	5	40	15	3	2	20	27	93	14	4	22	9	26	18	0

# Crear una tabla con el formato especificado
print("\nVehículos", end="\t")
zonas = ["A", "B", "C", "D"]

# Imprimir encabezados de entrada
for entrada in zonas:
    print(f"Entrada {entrada}", end="\t\t\t\t")
print()

# Imprimir subencabezados de salida
print("", end="\t")
for _ in range(4):
    for salida in zonas:
        print(f"Salida {salida}", end="\t")
print()

# Imprimir datos por cada tipo de vehículo
for clase_en, clase_es in class_names.items():
    print(f"{clase_es}", end="\t")
    for entrada in zonas:
        for salida in zonas:
            valor = conteo_por_ruta.get((entrada, salida), {}).get(clase_en, 0)
            print(f"{valor}", end="\t")
    print()

# Imprimir totales
print("Total", end="\t")
for entrada in zonas:
    for salida in zonas:
        total = sum(conteo_por_ruta.get((entrada, salida), {}).values())
        print(f"{total}", end="\t")
print()



Vehículos	Entrada A				Entrada B				Entrada C				Entrada D				
	Salida A	Salida B	Salida C	Salida D	Salida A	Salida B	Salida C	Salida D	Salida A	Salida B	Salida C	Salida D	Salida A	Salida B	Salida C	Salida D	
bicycle	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
bus	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
car	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
heavy_truck	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
light_truck	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
motorbike	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
Total	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
